# Processing a Flyte Dir in a Notebook

Demonstrates receiving a `Dir` input (as a path string) and
returning both a primitive and a `Dir` output.

In [ ]:
# Default values — papermill injects actual values after this cell
input_dir = None

In [ ]:
import os
import tempfile

from flyte.io import Dir

from flyteplugins.papermill import load_dir

# Reconstruct the Dir from the serialized path string
d = load_dir(input_dir)

# Download to local filesystem so we can list and process files
local_path = await d.download()
files = os.listdir(local_path)
file_count = len(files)
print(f"Directory has {file_count} files: {files}")

# Create a local output directory with a summary file, then upload
local_out = tempfile.mkdtemp()
summary_path = os.path.join(local_out, "summary.txt")
with open(summary_path, "w") as fh:
    fh.write(f"Processed {file_count} files:")
    fh.writelines(f"  - {name}" for name in sorted(files))

output_dir = await Dir.from_local(local_out)
print(f"Summary written, output dir at {output_dir.path}")

In [ ]:
from flyteplugins.papermill import record_outputs

record_outputs(file_count=file_count, output_dir=output_dir)